# Optimized Inference Deployment

In this section, we'll explore advanced frameworks for optimizing LLM deployments: Text Generation Inference (TGI), vLLM, and llama.cpp. These applications are primarily used in production environments to serve LLMs to users. This section focuses on how to deploy these frameworks in production rather than how to use them for inference on a single machine.

We'll cover how these tools maximize inference efficiency and simplify production deployments of Large Language Models.

## Framework Selection Guide

TGI, vLLM, and llama.cpp serve similar purposes but have distinct characteristics that make them better suited for different use cases. Let's look at the key differences between them, focusing on performance and integration.

### Memory Management and Performance

**TGI** is designed to be stable and predictable in production, using fixed sequence lengths to keep memory usage consistent. TGI manages memory using Flash Attention 2 and continuous batching techniques. This means it can process attention calculations very efficiently and keep the GPU busy by constantly feeding it work. The system can move parts of the model between CPU and GPU when needed, which helps handle larger models. 

> Flash Attention is a technique that optimizes the attention mechanism in transformer models by addressing memory bandwidth bottlenecks. As discussed earlier in [Chapter 1.8](/course/chapter1/8), the attention mechanism has quadratic complexity and memory usage, making it inefficient for long sequences.
> 
> The key innovation is in how it manages memory transfers between High Bandwidth Memory (HBM) and faster SRAM cache. Traditional attention repeatedly transfers data between HBM and SRAM, creating bottlenecks by leaving the GPU idle. Flash Attention loads data once into SRAM and performs all calculations there, minimizing expensive memory transfers. 
> 
> While the benefits are most significant during training, Flash Attention's reduced VRAM usage and improved efficiency make it valuable for inference as well, enabling faster and more scalable LLM serving.

**vLLM** takes a different approach by using PagedAttention. Just like how a computer manages its memory in pages, vLLM splits the model's memory into smaller blocks. This clever system means it can handle different-sized requests more flexibly and doesn't waste memory space. It's particularly good at sharing memory between different requests and reduces memory fragmentation, which makes the whole system more efficient.

> PagedAttention is a technique that addresses another critical bottleneck in LLM inference: KV cache memory management. As discussed in [Chapter 1.8](/course/chapter1/8), during text generation, the model stores attention keys and values (KV cache) for each generated token to reduce redundant computations. The KV cache can become enormous, especially with long sequences or multiple concurrent requests.
> 
> vLLM's key innovation lies in how it manages this cache:
> 
> 1. **Memory Paging**: Instead of treating the KV cache as one large block, it's divided into fixed-size "pages" (similar to virtual memory in operating systems).
> 2. **Non-contiguous Storage**: Pages don't need to be stored contiguously in GPU memory, allowing for more flexible memory allocation.
> 3. **Page Table Management**: A page table tracks which pages belong to which sequence, enabling efficient lookup and access.
> 4. **Memory Sharing**: For operations like parallel sampling, pages storing the KV cache for the prompt can be shared across multiple sequences.
> 
> The PagedAttention approach can lead to up to 24x higher throughput compared to traditional methods, making it a game-changer for production LLM deployments. If you want to go really deep into how PagedAttention works, you can read the [the guide from the vLLM documentation](https://docs.vllm.ai/en/latest/design/kernel/paged_attention.html).

**llama.cpp** is a highly optimized C/C++ implementation originally designed for running LLaMA models on consumer hardware. It focuses on CPU efficiency with optional GPU acceleration and is ideal for resource-constrained environments. llama.cpp uses quantization techniques to reduce model size and memory requirements while maintaining good performance. It implements optimized kernels for various CPU architectures and supports basic KV cache management for efficient token generation.

> Quantization in llama.cpp reduces the precision of model weights from 32-bit or 16-bit floating point to lower precision formats like 8-bit integers (INT8), 4-bit, or even lower. This significantly reduces memory usage and improves inference speed with minimal quality loss.
> 
> Key quantization features in llama.cpp include:
> 
> 1. **Multiple Quantization Levels**: Supports 8-bit, 4-bit, 3-bit, and even 2-bit quantization
> 2. **GGML/GGUF Format**: Uses custom tensor formats optimized for quantized inference
> 3. **Mixed Precision**: Can apply different quantization levels to different parts of the model
> 4. **Hardware-Specific Optimizations**: Includes optimized code paths for various CPU architectures (AVX2, AVX-512, NEON)
>
> This approach enables running billion-parameter models on consumer hardware with limited memory, making it perfect for local deployments and edge devices.

### Deployment and Integration

Let's move on to the deployment and integration differences between the frameworks.

**TGI** excels in enterprise-level deployment with its production-ready features. It comes with built-in Kubernetes support and includes everything you need for running in production, like monitoring through Prometheus and Grafana, automatic scaling, and comprehensive safety features. The system also includes enterprise-grade logging and various protective measures like content filtering and rate limiting to keep your deployment secure and stable.

**vLLM** takes a more flexible, developer-friendly approach to deployment. It's built with Python at its core and can easily replace OpenAI's API in your existing applications. The framework focuses on delivering raw performance and can be customized to fit your specific needs. It works particularly well with Ray for managing clusters, making it a great choice when you need high performance and adaptability.

**llama.cpp** prioritizes simplicity and portability. Its server implementation is lightweight and can run on a wide range of hardware, from powerful servers to consumer laptops and even some high-end mobile devices. With minimal dependencies and a simple C/C++ core, it's easy to deploy in environments where installing Python frameworks would be challenging. The server provides an OpenAI-compatible API while maintaining a much smaller resource footprint than other solutions.

## Getting Started

Let's explore how to use these frameworks for deploying LLMs, starting with installation and basic setup.

---
### TGI: Installation and Basic Setup
---

TGI is easy to install and use, with deep integration into the Hugging Face ecosystem.

First, launch the TGI server using Docker:

on linux:

```sh
docker run --gpus all \
    --shm-size 8g \
    -p 8080:80 \
    -v ~/.cache/huggingface:/data \
    ghcr.io/huggingface/text-generation-inference:latest \
    --model-id HuggingFaceTB/SmolLM2-360M-Instruct
```

on macOS:

```
docker run --platform linux/amd64 --gpus all \
    --shm-size 8g \
    -p 8080:80 \
    -v ~/.cache/huggingface:/data \
    ghcr.io/huggingface/text-generation-inference:latest \
    --model-id HuggingFaceTB/SmolLM2-360M-Instruct
```

Then interact with it using Hugging Face's InferenceClient:

In [1]:
from huggingface_hub import InferenceClient

client = InferenceClient(
    base_url="http://localhost:8080",
)

thread = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Tell me a story"}
]

response = client.chat_completion(
    messages=thread,
    max_tokens=128,
    temperature=0.7,
    top_p=0.95,
)

print("===== CHAT COMPLETION =====")
thread.append({"role": "assistant", "content": response.choices[0].message.content})
thread

===== CHAT COMPLETION =====


[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'Tell me a story'},
 {'role': 'assistant',
  'content': "In the misty land of Marr, where the sun barely reaches the horizon, there lived a wise and kind young girl named Aria. She was a curious and adventurous child, with a heart full of wonder and a mind full of questions. One day, while exploring the forest, Aria stumbled upon an ancient, mysterious-looking tree. As she reached out to touch the trunk, she felt an inexplicable pull towards it.\n\nWithout thinking, Aria reached out and touched the tree's trunk, feeling a surge of energy flow through her body. At that moment, a small, glowing rock was released from the"}]

---

Alternatively, you can use the OpenAI client:

In [2]:
!uv pip install -qq openai

In [3]:
from openai import OpenAI

# Initialize client pointing to TGI endpoint
client = OpenAI(
    base_url="http://localhost:8080/v1",  # Make sure to include /v1
    api_key="not-needed",  # TGI doesn't require an API key by default
)

thread = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Tell me a story"}
]

# Chat completion
response = client.chat.completions.create(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    messages=thread,
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)

print("===== CHAT COMPLETION =====")
thread.append({"role": "assistant", "content": response.choices[0].message.content})
thread

===== CHAT COMPLETION =====


[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'Tell me a story'},
 {'role': 'assistant',
  'content': 'In the land of Chronos, where time flowed like a river, there was once a great king named Tharros. He was a just ruler, beloved by his people, who lived for 800 years. But even the longest reign could not last forever, and one day, Tharros passed away.\n\nHis kingdom, Chronos, was left without a ruler. The people were afraid that without a strong leader, the kingdom would fall into chaos and disorder. So,'}]

---
### Llama: Installation and Basic Setup
---

llama.cpp is easy to install and use, requiring minimal dependencies and supporting both CPU and GPU inference.

First, install and build llama.cpp using following script:

---
```bash
#!/bin/bash

# Clone the repository
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp

# install cmake
sudo DEBIAN_FRONTEND=noninteractive apt-get update -y
sudo DEBIAN_FRONTEND=noninteractive apt-get install -y libssl-dev ccache cmake

# Build the project:
cmake .
make -j8 -s

# Download the SmolLM2-1.7B-Instruct-GGUF model
MODEL="SmolLM2-1.7B-Instruct-GGUF/resolve/main/smollm2-1.7b-instruct-q4_k_m.gguf"
wget --quiet "https://huggingface.co/HuggingFaceTB/$MODEL?download=true" -O "smollm2.gguf"

./bin/llama-server -m smollm2.gguf --host 0.0.0.0 --port 8080 -c 4096
```
---

Interact with the server using Hugging Face's InferenceClient:

In [4]:
from huggingface_hub import InferenceClient

client = InferenceClient(
    base_url="http://localhost:8080",
)

thread = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Tell me a story"}
]

response = client.chat_completion(
    messages=thread,
    max_tokens=128,
    temperature=0.7,
    top_p=0.95,
)

print("===== CHAT COMPLETION =====")
thread.append({"role": "assistant", "content": response.choices[0].message.content})
thread

===== CHAT COMPLETION =====


[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'Tell me a story'},
 {'role': 'assistant',
  'content': 'Once upon a time, in a land far, far away, there was a young, brave knight named Sir Edward. He was a knight with a kind heart, a strong arm, and a sharp mind. Sir Edward was known throughout his kingdom for his bravery and his desire to help those in need.\n\nOne day, a terrible dragon attacked the kingdom, causing great destruction and fear. The people were in danger, and they begged Sir Edward to save them. He, with his strong arm and sharp mind, led the knights in battle against the dragon.\n\nFor many days, they fought bravely, but the dragon was'}]

---

Alternatively, you can use the OpenAI client:

In [5]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="not-needed",
)

thread = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Tell me a story"}
]

# Chat completion
response = client.chat.completions.create(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    messages=thread,
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)

print("===== CHAT COMPLETION =====")
thread.append({"role": "assistant", "content": response.choices[0].message.content})
thread

===== CHAT COMPLETION =====


[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'Tell me a story'},
 {'role': 'assistant',
  'content': 'Once upon a time, there was a young girl named Lily. She lived in a small village surrounded by rolling hills and dense forests. Lily was a curious and adventurous child, always seeking new adventures and exciting tales to hear.\n\nOne sunny afternoon, Lily decided to venture into the nearby forest. She packed some food and water for the day and set off on her journey, eager to explore the unknown. As she walked deeper into the forest, she stumbled upon a hidden cave behind a thick veil'}]

---
### vLLM: Installation and Basic Setup
---

vLLM is easy to install and use, with both OpenAI API compatibility and a native Python interface.

First, install vllm:

In [6]:
!uv pip install -qq vllm

Second, launch the vLLM OpenAI-compatible server:

```
source .venv/bin/activate && python -m vllm.entrypoints.openai.api_server \
    --model HuggingFaceTB/SmolLM2-360M-Instruct \
    --host 0.0.0.0 \
    --port 8080
```

---

Then interact with it using Hugging Face's InferenceClient:

In [7]:
from huggingface_hub import InferenceClient

client = InferenceClient(
    base_url="http://localhost:8080/v1",
)

thread = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Tell me a story"}
]

# Chat completion
response = client.chat.completions.create(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    messages=thread,
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)

print("===== CHAT COMPLETION =====")
thread.append({"role": "assistant", "content": response.choices[0].message.content})
thread

===== CHAT COMPLETION =====


[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'Tell me a story'},
 {'role': 'assistant',
  'content': 'Once upon a time, in a small village nestled between a vast forest and a river, lived a young girl named Aria. She was known for her curious nature and had a knack for solving puzzles. One day, while exploring the woods near her home, she stumbled upon an ancient, mysterious-looking box. The box had a strange symbol on it that seemed to shimmer in the light. \n\nAria approached the box cautiously, her heart racing with excitement. She reached out a tre'}]

Alternatively, you can use the OpenAI client:

In [8]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="not-needed",
)

thread = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Tell me a story"}
]

# Chat completion
response = client.chat.completions.create(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    messages=thread,
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)

print("===== CHAT COMPLETION =====")
thread.append({"role": "assistant", "content": response.choices[0].message.content})
thread

===== CHAT COMPLETION =====


[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'Tell me a story'},
 {'role': 'assistant',
  'content': 'Once upon a time, in a faraway land, there lived a wise and kind King who was known throughout the kingdom for his love of storytelling. He had a loyal and mischievous dog named Blinky who would often come to him with tales of adventure, magic, and excitement.\n\nOne day, as King Arthur sat on his throne, he received a strange letter from an unknown sender. The letter was short, but it was filled with words of wisdom and advice. It spoke of the importance'}]